<a href="https://colab.research.google.com/github/mhsahel/Making_ML_models_and_Evaluation/blob/main/Mental_Health_Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import   seaborn as sns
import pandas as pd

In [ ]:
df = pd.read_csv("mental_health_store.csv")

In [ ]:
print(f"Rows :{df.shape[0]}\nColumns: {df.shape[1]}")

In [ ]:
# First 5 datas
df.head()

In [ ]:
# check the duplicated
df.duplicated().sum()

In [ ]:
# check those duplicat rows
df[df.duplicated()]

In [ ]:
# check nulls values
df.isna().sum()

In [ ]:
# check descrivtive
df.describe()

In [ ]:
# check all data types
df.info()

# perform the EDA(exploratiry data analysis)

In [ ]:
# check the target column first by using hist plot
sns.histplot(df['Mental_Health_Score'], kde = True)

# Check the correlations of all numerical columns

In [ ]:
# using heatmap with over all the numerical columns
sns.heatmap(df.corr(numeric_only=True), annot=True)

In [ ]:
# select all categorical column
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
cat_cols

In [ ]:
df.info()

In [ ]:
# relationship between stress_level vs mental_health_score
stress_lvl = df['Stress_Level'].unique().tolist()

In [ ]:
#check the outliers with bloxplot
sns.boxplot(data = df, x = 'Stress_Level', y = 'Mental_Health_Score')

In [ ]:
# check daily usage with the mental score
sns.scatterplot(data = df , x ='Mental_Health_Score', y = 'Avg_Daily_Usage_Hours')

In [ ]:
# check sleep hours  with the mental score
sns.scatterplot(data = df , x ='Mental_Health_Score', y = 'Sleep_Hours_Per_Night')

In [ ]:
df['Most_Used_Platform'].value_counts()

In [ ]:
# check which platform are mostly used
plt.figure(figsize=(10,8))
sns.countplot(order =  df['Most_Used_Platform'].value_counts().index , x = df['Most_Used_Platform'])
plt.tight_layout()

In [ ]:
# checking study hours vs mental_health_score
sns.scatterplot(data = df , x='Mental_Health_Score', y = 'Study_Hours')

# check the outliers  using IQR methods

In [ ]:
numeric_features = df.select_dtypes(include = 'number')
q1= numeric_features.quantile(0.25)
q3 = numeric_features.quantile(0.75)
IQR =q3 - q1

In [ ]:
lower_bound = q1 - 1.5 * IQR
upper_bound = q3 + 1.5 * IQR

In [ ]:
outliers = (numeric_features > upper_bound) | (numeric_features<lower_bound)

# Data Cleaning Process

In [ ]:
# finding the duplicates from the my dataset

print(f"The number of duplicates = {df.duplicated().sum()}")

In [ ]:
# checking which two are duplicates for further droping if needed
df[df.duplicated(keep = False)]

In [ ]:
#drop the duplicates
df.drop_duplicates(inplace = True)


In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)

In [ ]:
df.describe()

In [ ]:

num_cols = df.select_dtypes(include='number')
num_cols.skew()

In [ ]:
numeric_col = df.select_dtypes(include = 'number').columns.tolist()
numeric_col

In [ ]:
fig , axes = plt.subplots(ncols = 4, nrows = 2 , figsize =(15,8))
axes = axes.flatten()
for i , col in enumerate(numeric_col):
  sns.histplot(data = df , ax=axes[i], x = col, kde = True)
  axes[i].set_title("Distribution")
plt.tight_layout()
plt.show()

#Feature Engineering


In [ ]:
top_countries = df['Country'].value_counts().index[1:11].tolist()
def select_countries(country):
  if country in top_countries:
    return country
  else:
    return 'Other'

In [ ]:
top_countries

In [ ]:
df['Country'].value_counts()

In [ ]:

df['Country'] = df['Country'].apply(select_countries)

In [ ]:

df['Country'].value_counts()

# Train Test split

In [ ]:

from sklearn.model_selection import train_test_split

skwed_col         = ['Study_Hours']
other_numeric_cols = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks","Physical_Activity_Hours",
                      "Sleep_Hours_Per_Night"]
ordinal_col        = ['Stress_Level']
normal_col         = ["Gender", "Academic_Level", "Most_Used_Platform", "Purpose_Of_Use", "Country"]


feature_col = skwed_col + other_numeric_cols + ordinal_col + normal_col
X = df[feature_col]
y = df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)


In [ ]:
df['Stress_Level'].value_counts()

# prepocessing the columns with creating Pipelins

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder,OneHotEncoder
skew_pipeline = make_pipeline(FunctionTransformer(np.log1p),StandardScaler())

other_num_pipeline = make_pipeline(StandardScaler())

ordinal_pipeline = make_pipeline(
OrdinalEncoder(categories=[['Low', 'Medium', 'High', 'Very High']]))

one_hot_pipeline = make_pipeline(OneHotEncoder(handle_unknown='ignore'))

In [ ]:
from sklearn.compose import ColumnTransformer
preprocess= ColumnTransformer(transformers=[

                                     ('skewed_pipelin', skew_pipeline,skwed_col),
                                     ('other_col_pipeline',other_num_pipeline,other_numeric_cols),
                                     ('ordinal_pipeline',ordinal_pipeline,ordinal_col),
                                     ('one_hot_pipeline',one_hot_pipeline,normal_col)
])
preprocess

# Base -: Model Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
lr_pipeline = make_pipeline(preprocess, LinearRegression())
lr_pipeline

In [ ]:
lr_pipeline.fit(X_train , y_train)

In [ ]:
lr_pred_test = lr_pipeline.predict(X_test)
lr_pred_train = lr_pipeline.predict(X_train)


# Checking Accuracy Of The Model

In [ ]:
lr_r2_testing=r2_score(y_test , lr_pred_test)
lr_r2_training=r2_score(y_train , lr_pred_train)

print(f"Training R2_Accuracy: {lr_r2_training}")
print(f"Testing R2_Accuracy: {lr_r2_testing}")

In [ ]:
rf_mse_train = mean_squared_error(y_train , lr_pred_train)
rf_mse_test = mean_squared_error(y_test , lr_pred_test)
print(f"Training Mean Squared Error: {rf_mse_train }")
print(f"Testing Mean Squared Error: {rf_mse_test}")

In [ ]:
rf_mae_train = mean_absolute_error(y_train , lr_pred_train)
rf_mae_test = mean_absolute_error(y_test , lr_pred_test)
print(f"Training Mean Absolute Error: {rf_mae_train }")
print(f"Testing Mean Absolute Error: {rf_mae_test}")

# Random_Forest Model


In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf_pipe= make_pipeline(preprocess , RandomForestRegressor(random_state=42))


In [ ]:
rf_pipe.fit(X_train,y_train)

In [ ]:
rf_pred_train = rf_pipe.predict(X_train)
rf_pred_test = rf_pipe.predict(X_test)

In [ ]:
rf_r2_score_train = r2_score(y_train , rf_pred_train)
rf_r2_score_test = r2_score(y_test , rf_pred_test)

print(f"Training R2_Accuracy for random forest: {rf_r2_score_train}\n")

print(f"Testing R2_Accuracy for random forest: {rf_r2_score_test}\n")
rf_mse_train = mean_squared_error(y_train , rf_pred_train)
rf_mse_test = mean_squared_error(y_test , rf_pred_test)

print(f"Training Mean Squared Error for random forest: {rf_mse_train}\n")

print(f"Testing Mean Squared Error for random forest: {rf_mse_test}\n")
rf_mae_train = mean_absolute_error(y_train , rf_pred_train)
rf_mae_test = mean_absolute_error(y_test , rf_pred_test)

print(f"Training Mean Absolute Error for random forest: {rf_mae_train}\n")

print(f"Testing Mean Absolute Error for random forest: {rf_mae_test}")

In [ ]:
params = {'randomforestregressor__n_estimators':[100,300,500],'randomforestregressor__max_depth':[10,6,9],'randomforestregressor__max_leaf_nodes':[2,4,6],'randomforestregressor__min_samples_leaf':[2,3,4],'randomforestregressor__min_samples_split':[2,5,10]}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(estimator=rf_pipe, param_distributions=params,n_jobs=-1,n_iter=10,cv=5,scoring='r2')
rs.fit(X_train , y_train)

In [ ]:
rf_rand_pred_train = rs.predict(X_train)
rf_rand__pred_test = rs.predict(X_test)

In [ ]:

print(f"Training R2_Accuracy for random forest after tuning: {r2_score(y_train ,rf_rand_pred_train)}\n")

print(f"Testing R2_Accuracy for random forest after tuning: {r2_score(y_test , rf_rand__pred_test )}\n")


print(f"Training Mean Squared Error for random forest after tuning: {mean_squared_error(y_train , rf_rand_pred_train)}\n")

print(f"Testing Mean Squared Error for random forest after tuning: {mean_squared_error(y_test ,rf_rand__pred_test )}\n")



print(f"Training Mean Absolute Error for random forest after tuning: {mean_absolute_error(y_train , rf_rand_pred_train)}\n")

print(f"Testing Mean Absolute Error for random forest after tuning: {mean_absolute_error(y_test , rf_rand__pred_test )}")

In [ ]:
rf_tuned_mae= mean_absolute_error(y_test , rf_rand__pred_test )

In [ ]:
print(f"Best all parameters{rs.best_params_}")

Choosing Best Model

In [ ]:
from sklearn.metrics import mean_squared_error

# Calculate RMSE for Linear Regression
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred_test))

# Calculate RMSE for default Random Forest
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred_test))

# Calculate RMSE for tuned Random Forest
rf_tuned_preds          = rs.best_estimator_.predict(X_test)
rf_tuned_rmse           = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))
rf_tuned_training_preds = rs.best_estimator_.predict(X_train)
r2_tuned_training       = r2_score(y_train, rf_tuned_training_preds)
rf_tuned_mae            = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2             = r2_score(y_test, rf_tuned_preds)

# Create a DataFrame to consolidate results

# Create a DataFrame to consolidate results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_score_test, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_score_train , r2_tuned_training],
    'MAE': [rf_mae_test, rf_mae_test, rf_tuned_mae],  # rf_mae_test is used for both based on your previously executed cells
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

print(results)
